In [1]:
import requests
import pandas as pd
import time
import os
from IPython.display import display

In [2]:
# ==========================================
# 1. API AND SEARCH CONFIGURATION
# ==========================================

# Read the API Key from the text file parsing the "API_Google=" format
try:
    with open('API_google.txt', 'r') as file:
        raw_content = file.read().strip()
        
        # Check if the format contains '=' and split it to get only the key
        if '=' in raw_content:
            API_KEY = raw_content.split('=', 1)[1].strip()
        else:
            API_KEY = raw_content # Fallback just in case it's only the key
            
except FileNotFoundError:
    raise FileNotFoundError("Error: The file 'API_google.txt' was not found. Make sure it is in the same directory as this notebook.")

# Central coordinates of Boca Chica, Dominican Republic
LAT = 18.4497 
LNG = -69.6006
RADIUS = 3000 # 3 km radius

In [3]:
# List of key industries to map
industry_types = [
    # Tourism
    'restaurant', 'lodging', 'bar', 'cafe', 'night_club', 'casino', 'travel_agency', 'tourist_attraction',
    
    # Commerce
    'store', 'supermarket', 'grocery_or_supermarket', 'convenience_store', 'bakery', 'liquor_store', 'clothing_store', 'shoe_store', 'electronics_store', 'furniture_store', 'home_goods_store', 'jewelry_store',
    
    # Health
    'health', 'pharmacy', 'drugstore', 'hospital', 'doctor', 'dentist', 'physiotherapist', 'gym', 'spa', 'beauty_salon',
    
    # services
    'bank', 'atm', 'post_office', 'hardware_store', 'laundry', 'car_repair', 'car_wash', 'car_rental', 'car_dealer', 'gas_station', 'real_estate_agency', 'lawyer', 'accounting',
    
    # infastructure
    'local_government_office', 'police', 'school', 'church', 'veterinary_care', 'park', 'parking'
]

endpoint_url = "https://maps.googleapis.com/maps/api/place/nearbysearch/json"
extracted_businesses = []

print("Starting Boca Chica commercial mapping...\n")

Starting Boca Chica commercial mapping...



In [4]:
# ==========================================
# 2. EXTRACTION AND PAGINATION ENGINE
# ==========================================
for industry in industry_types:
    print(f"Searching for businesses in the industry: {industry.upper()}...")
    
    params = {
        'location': f"{LAT},{LNG}",
        'radius': RADIUS,
        'type': industry,
        'key': API_KEY
    }
    
    while True:
        response = requests.get(endpoint_url, params=params)
        data = response.json()
        
        # Validate API Key errors
        if data.get('status') == 'REQUEST_DENIED':
            raise ValueError(f"API Error: {data.get('error_message', 'Check your API Key')}")
            
        results = data.get('results', [])
        
        # Process each found business
        for place in results:
            raw_categories = place.get('types', [])
            main_industry = raw_categories[0].replace('_', ' ').title() if raw_categories else 'Unknown'
            extra_categories = ", ".join([c.replace('_', ' ').title() for c in raw_categories[1:]])
            
            business = {
                'Place_ID': place.get('place_id'),
                'Business_Name': place.get('name'),
                'Target_Industry': industry.title(),
                'Main_Classification': main_industry,
                'Sub_Categories': extra_categories,
                'Price_Level_(0-4)': place.get('price_level', 'Not available'),
                'Average_Rating': place.get('rating', 0),
                'Total_Reviews': place.get('user_ratings_total', 0),
                'Address': place.get('vicinity', 'Not specified'),
                'Latitude': place.get('geometry', {}).get('location', {}).get('lat'),
                'Longitude': place.get('geometry', {}).get('location', {}).get('lng')
            }
            extracted_businesses.append(business)
        
        # Pagination handling
        next_page_token = data.get('next_page_token')
        if next_page_token:
            time.sleep(2) # Required pause by Google before using the token
            params['pagetoken'] = next_page_token
        else:
            break # Exit the while loop if there are no more pages

#confirmation message 
print("\n--- Data extraction for all industries completed successfully! ---")

Searching for businesses in the industry: RESTAURANT...
Searching for businesses in the industry: LODGING...
Searching for businesses in the industry: BAR...
Searching for businesses in the industry: CAFE...
Searching for businesses in the industry: NIGHT_CLUB...
Searching for businesses in the industry: CASINO...
Searching for businesses in the industry: TRAVEL_AGENCY...
Searching for businesses in the industry: TOURIST_ATTRACTION...
Searching for businesses in the industry: STORE...
Searching for businesses in the industry: SUPERMARKET...
Searching for businesses in the industry: GROCERY_OR_SUPERMARKET...
Searching for businesses in the industry: CONVENIENCE_STORE...
Searching for businesses in the industry: BAKERY...
Searching for businesses in the industry: LIQUOR_STORE...
Searching for businesses in the industry: CLOTHING_STORE...
Searching for businesses in the industry: SHOE_STORE...
Searching for businesses in the industry: ELECTRONICS_STORE...
Searching for businesses in the i

In [6]:
# ==========================================
# 3. DATA CLEANING, EXPORT AND DISPLAY
# ==========================================
print("\nStructuring data and removing duplicates...")
df_boca_chica = pd.DataFrame(extracted_businesses)

if not df_boca_chica.empty:
    df_boca_chica = df_boca_chica.drop_duplicates(subset=['Place_ID'])
    df_boca_chica = df_boca_chica.sort_values(by=['Target_Industry', 'Total_Reviews'], ascending=[True, False])
    
    file_name = 'Boca_Chica_Commercial_Census.xlsx'
    df_boca_chica.to_excel(file_name, index=False, engine='openpyxl')
    
    print(f"Success! {len(df_boca_chica)} unique businesses have been structured in '{file_name}'.")
    
    # Display the first 5 rows elegantly in Jupyter
    print("\nData Preview:")
    display(df_boca_chica.head())
else:
    print("No data was found to extract.")


Structuring data and removing duplicates...
Success! 693 unique businesses have been structured in 'Boca_Chica_Commercial_Census.xlsx'.

Data Preview:


,Place_ID,Business_Name,Target_Industry,Main_Classification,Sub_Categories,Price_Level_(0-4),Average_Rating,Total_Reviews,Address,Latitude,Longitude
727,ChIJDfY6Avd_r44REWqUIsQAJQs,Impuestos Internos,Accounting,Local Government Office,"Accounting, Finance, Point Of Interest, Establ...",Not available,4.9,8,"Plaza Villa Florencia, Avenue 20 de Diciembre,...",18.449174,-69.606467
728,ChIJNwzJ-zh-r44RzNLvYgaJjPg,MELC Solucines Agiles,Accounting,Accounting,"Finance, Point Of Interest, Establishment",Not available,0.0,0,"Avenue Duarte 28, Boca Chica",18.447673,-69.606161
623,ChIJtQyC_kZ-r44R6hx4hFhBgMQ,Cajero Automatico Banco BHD,Atm,Atm,"Finance, Point Of Interest, Establishment",Not available,3.0,3,"F93V+6GM, Boca Chica",18.453098,-69.606190
626,ChIJG62zgzd-r44RBZ1MYpzssIk,Bank Machine,Atm,Atm,"Finance, Point Of Interest, Establishment",Not available,4.7,3,"C9XQ+5X9, Avenue Duarte, Boca Chica",18.447911,-69.610030
628,ChIJS3AOhK5_r44Rv-hU7JC2oL8,ATM Cajero Triinet Whala! Boca Chica,Atm,Atm,"Finance, Point Of Interest, Establishment",Not available,4.7,3,"C9WQ+RR Whala!, Boca Chica",18.447058,-69.610408
